In [ ]:
import glob
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import pipeline
from PIL import Image

import wandb
import random
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torch.nn as nn

!pip install onnxruntime
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import time
import importlib
import torch.hub
import sys

from scipy.ndimage import binary_dilation
import matplotlib.gridspec as gridspec

In [ ]:
INPUT_PATH = os.environ.get('KITTI_DATA_DIR', '/kaggle/input/datasets')
OUTPUT_PATH = os.environ.get('OUTPUT_DIR', '/kaggle/working')
SAVE_PATH = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned.pth')

# training config
MODEL_NAME = "DPT_Large"
EPOCHS = 15
BATCH_SIZE = 2 # start with 2 for DPT_Large
LR = 1e-4
TRAIN_SIZE = 160
VAL_SIZE = 40
LOSS = "HuberLoss"
GRID_RES = 0.2
X_RANGE = (-25, 35)
Z_RANGE = (3, 80)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using:", device)

In [ ]:
# --- SET THESE TO MATCH YOUR KITTI DIRECTORY LAYOUT ---
# On Kaggle: os.path.join(INPUT_PATH, '<dataset-slug>', 'kitti-stereo-2015', 'data', 'data_scene_flow')
# Locally:   something like './data/kitti/data_scene_flow'
KITTI_ROOT = os.path.join(INPUT_PATH, 'kitti-stereo-2015', 'data', 'data_scene_flow')
calib_path = os.path.join(KITTI_ROOT, 'data_scene_flow_calib', 'training', 'calib_cam_to_cam', '000000.txt')


In [ ]:
for root, dirs, files in os.walk(data_root):
    level = root.replace(data_root, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files[:3]:  # only first 3 files per folder
        print(f'{indent}  {file}')

In [ ]:
def get_kitti_paths(split='training'):
    img2_dir = Path(KITTI_ROOT) / split / 'image_2'
    img3_dir = Path(KITTI_ROOT) / split / 'image_3'
    disp_dir = Path(KITTI_ROOT) / split / 'disp_occ_0'

    # get all left images, only _10 files (first frame of each scene)
    left_imgs = sorted([f for f in img2_dir.glob('*_10.png')])

    triplets = []
    for left_path in left_imgs:
        name = left_path.name  # e.g. 000110_10.png
        right_path = img3_dir / name
        disp_path = disp_dir / name

        # only include if all three exist
        if right_path.exists() and disp_path.exists():
            triplets.append((left_path, right_path, disp_path))

    return triplets

In [ ]:
triplets = get_kitti_paths('training')
print(f"found {len(triplets)} valid triplets")
print(triplets[0])

In [ ]:
left, right, disp_path = triplets[0]

left_img = cv2.imread(str(left))
left_img = cv2.cvtColor(left_img, cv2.COLOR_BGR2RGB)

right_img = cv2.imread(str(right))
right_img = cv2.cvtColor(right_img, cv2.COLOR_BGR2RGB)

# KITTI disparity is stored as uint16, divide by 256 to get real values
disp = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
disp[disp == 0] = np.nan  # 0 means no ground truth

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].imshow(left_img)
axes[0].set_title('Left Image')
axes[1].imshow(right_img)
axes[1].set_title('Right Image')
axes[2].imshow(disp, cmap='plasma')
axes[2].set_title('Ground Truth Disparity')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'l_r_gt_scene0.png'), dpi=150, bbox_inches='tight')
plt.show()

print("disparity min/max:", np.nanmin(disp), np.nanmax(disp))

In [ ]:
with open(calib_path) as f:
    lines = f.readlines()

for line in lines[:20]:
    print(line.strip())

In [ ]:
def read_calib(calib_path):
    with open(calib_path) as f:
        lines = f.readlines()

    calib = {}
    for line in lines:
        key, val = line.strip().split(':', 1)
        try:
            calib[key] = np.array([float(x) for x in val.strip().split()])
        except ValueError:
            continue  # skip non-numeric lines like calib_time

    P0 = calib['P_rect_00'].reshape(3, 4)
    P1 = calib['P_rect_01'].reshape(3, 4)

    f = P0[0, 0]
    B = -P1[0, 3] / P1[0, 0]

    return f, B

def read_calib_full(calib_path):
    with open(calib_path) as f:
        lines = f.readlines()
    calib = {}
    for line in lines:
        key, val = line.strip().split(':', 1)
        try:
            calib[key] = np.array([float(x) for x in val.strip().split()])
        except ValueError:
            continue
    P0 = calib['P_rect_00'].reshape(3, 4)
    f = P0[0, 0]
    cx = P0[0, 2]
    cy = P0[1, 2]
    B = -calib['P_rect_01'].reshape(3, 4)[0, 3] / f
    return f, B, cx, cy

In [ ]:
f, B = read_calib(calib_path)
print(f"focal length: {f:.2f} px")
print(f"baseline: {B:.4f} m")

In [ ]:
def compute_stereo(left_path, right_path):
    left = cv2.imread(str(left_path))
    right = cv2.imread(str(right_path))

    left_gray = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)
    right_gray = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)

    # StereoBM
    bm = cv2.StereoBM_create(numDisparities=128, blockSize=11)
    bm.setSpeckleWindowSize(80)
    bm.setSpeckleRange(32)
    bm.setUniquenessRatio(5)
    bm.setMinDisparity(0)
    disp_bm = bm.compute(left_gray, right_gray).astype(np.float32) / 16.0
    disp_bm[disp_bm <= 0] = np.nan

    # StereoSGBM
    sgbm = cv2.StereoSGBM_create(
        minDisparity=0,
        numDisparities=128,
        blockSize=11,
        P1=8*3*11**2,
        P2=32*3*11**2,
        disp12MaxDiff=1,
        uniquenessRatio=5,
        speckleWindowSize=80,
        speckleRange=32,
        mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
    )
    disp_sgbm = sgbm.compute(left_gray, right_gray).astype(np.float32) / 16.0
    disp_sgbm[disp_sgbm <= 0] = np.nan

    return disp_bm, disp_sgbm

In [ ]:
# test on first triplet
left_path, right_path, disp_path = triplets[0]
disp_bm, disp_sgbm = compute_stereo(left_path, right_path)

# load gt
disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
disp_gt[disp_gt == 0] = np.nan

left_img = cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB)

# visualize
fig, axes = plt.subplots(2, 2, figsize=(20, 8))

axes[0][0].imshow(left_img)
axes[0][0].set_title('Left Image')
axes[0][1].imshow(disp_gt, cmap='plasma', vmin=0, vmax=80)
axes[0][1].set_title('Ground Truth Disparity')
axes[1][0].imshow(disp_bm, cmap='plasma', vmin=0, vmax=80)
axes[1][0].set_title('StereoBM')
axes[1][1].imshow(disp_sgbm, cmap='plasma', vmin=0, vmax=80)
axes[1][1].set_title('StereoSGBM')

for row in axes:
    for ax in row:
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'bm_sgbm_scene0.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def disp_to_depth(disp, f, B):
    with np.errstate(divide='ignore', invalid='ignore'):
        depth = f * B / disp
        depth[depth <= 0] = np.nan
        depth[depth > 80] = np.nan # KITTI max range is 80m
    return depth

In [ ]:
f, B = read_calib(calib_path)
depth_bm = disp_to_depth(disp_bm, f, B)
depth_sgbm = disp_to_depth(disp_sgbm, f, B)

print("depth range BM:", np.nanmin(depth_bm), np.nanmax(depth_bm))
print("depth range SGBM:", np.nanmin(depth_sgbm), np.nanmax(depth_sgbm))

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(20, 12))

for i in range(5):
    left_path, right_path, disp_path = triplets[i]
    disp_bm, disp_sgbm = compute_stereo(left_path, right_path)

    depth_bm = disp_to_depth(disp_bm, f, B)
    depth_sgbm = disp_to_depth(disp_sgbm, f, B)

    # load gt depth from gt disparity
    disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
    disp_gt[disp_gt == 0] = np.nan
    depth_gt = disp_to_depth(disp_gt, f, B)

    left_img = cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB)

    axes[i][0].imshow(left_img)
    axes[i][0].set_title(f'Scene {i} - Left')
    axes[i][1].imshow(depth_gt, cmap='plasma', vmin=0, vmax=80)
    axes[i][1].set_title('GT Depth')
    axes[i][2].imshow(depth_bm, cmap='plasma', vmin=0, vmax=80)
    axes[i][2].set_title('BM Depth')
    axes[i][3].imshow(depth_sgbm, cmap='plasma', vmin=0, vmax=80)
    axes[i][3].set_title('SGBM Depth')

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'bm_sgbm_5scenes.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
left_path, right_path, disp_path = triplets[0]
disp_bm, disp_sgbm = compute_stereo(left_path, right_path)
depth_bm = disp_to_depth(disp_bm, f, B)
depth_sgbm = disp_to_depth(disp_sgbm, f, B)

In [ ]:
# load MiDaS
midas_small = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small')
midas_small.eval()

transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform_small = transforms.small_transform

# move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
midas_small.to(device)
print('using:', device)

# dpt large
midas_large = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
transform = transforms.dpt_transform
midas_large.to(device)
midas_large.eval()

In [ ]:
def run_midas(image_path, model, transform, device):
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    input_tensor = transform(img_rgb).to(device)

    with torch.no_grad():
        prediction = model(input_tensor)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode='bicubic',
            align_corners=False
        ).squeeze()

    depth = prediction.cpu().numpy()
    # INVERT - DPT outputs inverse depth (high = close, low = far)
    depth = 1.0 / (depth + 1e-8)
    return depth

def run_midas_finetuned(image_path, model, transform, device):
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_tensor = transform(img_rgb).to(device)
    
    with torch.no_grad():
        prediction = model(input_tensor)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_rgb.shape[:2],
            mode='bicubic',
            align_corners=False
        ).squeeze()
    
    depth = prediction.cpu().numpy()
    # NO inversion - finetuned model outputs metric depth directly
    return depth

In [ ]:
# test on first image
left_path, right_path, disp_path = triplets[0]
midas_depth_small = run_midas(left_path, midas_small, transform_small, device)
midas_depth = run_midas(left_path, midas_large, transform, device)

print("MiDaS Small output shape:", midas_depth_small.shape)
print("MiDaS Small output range:", midas_depth_small.min(), midas_depth_small.max())

print("DPT Large output shape:", midas_depth.shape)
print("DPT Large output range:", midas_depth.min(), midas_depth.max())

In [ ]:
def median_scale_align(pred, gt):
    mask = (gt > 0) & np.isfinite(gt) & np.isfinite(pred) & (pred > 0)
    scale = np.median(gt[mask]) / np.median(pred[mask])
    aligned = pred * scale
    aligned = np.clip(aligned, 0, 80)
    return aligned

In [ ]:
# load gt depth
disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
disp_gt[disp_gt == 0] = np.nan
depth_gt = disp_to_depth(disp_gt, f, B)

# align
midas_aligned_small = median_scale_align(midas_depth_small, depth_gt)
midas_aligned = median_scale_align(midas_depth, depth_gt)
print("aligned range small:", midas_aligned_small.min(), midas_aligned_small.max())
print("aligned range large:", midas_aligned.min(), midas_aligned.max())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 8))

axes[0][0].imshow(cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB))
axes[0][0].set_title('Left Image')
axes[0][1].imshow(depth_gt, cmap='plasma', vmin=0, vmax=80)
axes[0][1].set_title('GT Depth')
axes[1][0].imshow(midas_aligned_small, cmap='plasma', vmin=0, vmax=80)
axes[1][0].set_title('MiDaS Small')
axes[1][1].imshow(midas_aligned, cmap='plasma', vmin=0, vmax=80)
axes[1][1].set_title('DPT Large')

for row in axes:
    for ax in row:
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'midas_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
depth_anything = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 # GPU
)

In [ ]:
def run_depth_anything(image_path, pipeline):
    img = Image.open(str(image_path)).convert('RGB')
    result = pipeline(img)
    depth = np.array(result['predicted_depth'].squeeze()).astype(np.float32)
    depth = np.clip(depth, 0.1, None)  # clip to avoid div by zero and negatives
    depth = 1.0 / (depth + 1e-8)  # invert
    return depth

In [ ]:
da_raw = run_depth_anything(triplets[0][0], depth_anything)
plt.imshow(da_raw, cmap='plasma')
plt.colorbar()
plt.axis('off')
plt.title('DepthAnything raw')
plt.savefig(os.path.join(OUTPUT_PATH, 'da_raw.png'), dpi=150, bbox_inches='tight')
plt.show()
print("range:", da_raw.min(), da_raw.max())

In [ ]:
da_raw = np.array(depth_anything(Image.open(str(triplets[0][0])).convert('RGB'))['predicted_depth'].squeeze())
print("min:", da_raw.min(), "max:", da_raw.max())
print("negative pixels:", (da_raw < 0).sum())
print("zero pixels:", (da_raw == 0).sum())

In [ ]:
# test
da_depth = run_depth_anything(left_path, depth_anything)
print("DepthAnything output shape:", da_depth.shape)
print("DepthAnything output range:", da_depth.min(), da_depth.max())

In [ ]:
da_aligned = median_scale_align(da_depth, depth_gt)
print("DA aligned range:", da_aligned.min(), da_aligned.max())

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(20, 12))

axes[0][0].imshow(cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB))
axes[0][0].set_title('Left Image')
axes[0][1].imshow(depth_gt, cmap='plasma', vmin=0, vmax=80)
axes[0][1].set_title('GT Depth')

axes[1][0].imshow(depth_bm, cmap='plasma', vmin=0, vmax=80)
axes[1][0].set_title('StereoBM')
axes[1][1].imshow(depth_sgbm, cmap='plasma', vmin=0, vmax=80)
axes[1][1].set_title('StereoSGBM')

axes[2][0].imshow(midas_aligned, cmap='plasma', vmin=0, vmax=80)
axes[2][0].set_title('DPT Large')
axes[2][1].imshow(da_aligned, cmap='plasma', vmin=0, vmax=80)
axes[2][1].set_title('DepthAnything V2')

for row in axes:
    for ax in row:
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'depth_comparison_scene0.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def compute_metrics(pred, gt):
    """
    pred: predicted depth map (H, W) in meters
    gt: ground truth depth map (H, W) in meters
    returns: dict of metrics
    """
    # only evaluate where GT is valid
    mask = (gt > 0) & np.isfinite(gt) & np.isfinite(pred) & (pred > 0)

    if mask.sum() == 0:
        return None

    pred_masked = pred[mask]
    gt_masked = gt[mask]

    # MAE
    mae = np.mean(np.abs(pred_masked - gt_masked))

    # RMSE
    rmse = np.sqrt(np.mean((pred_masked - gt_masked) ** 2))

    # AbsRel
    absrel = np.mean(np.abs(pred_masked - gt_masked) / gt_masked)

    # threshold accuracy
    ratio = np.maximum(pred_masked / gt_masked, gt_masked / pred_masked)
    delta1 = np.mean(ratio < 1.25)
    delta2 = np.mean(ratio < 1.25 ** 2)
    delta3 = np.mean(ratio < 1.25 ** 3)

    return {
        'mae': mae,
        'rmse': rmse,
        'absrel': absrel,
        'delta1': delta1,
        'delta2': delta2,
        'delta3': delta3
    }
    
def average_metrics(metrics_list):
    """average a list of metric dicts"""
    metrics_list = [m for m in metrics_list if m is not None]
    if not metrics_list:
        return None

    keys = metrics_list[0].keys()
    return {k: np.mean([m[k] for m in metrics_list]) for k in keys}

def print_results_table(results_dict):
    """
    results_dict: {'method_name': avg_metrics_dict}
    """
    print(f"{'Method':<20} {'MAE':>8} {'RMSE':>8} {'AbsRel':>8} {'δ<1.25':>8} {'δ<1.25²':>8} {'δ<1.25³':>8}")
    print("-" * 76)
    for method, metrics in results_dict.items():
        print(
            f"{method:<20} "
            f"{metrics['mae']:>8.3f} "
            f"{metrics['rmse']:>8.3f} "
            f"{metrics['absrel']:>8.3f} "
            f"{metrics['delta1']:>8.3f} "
            f"{metrics['delta2']:>8.3f} "
            f"{metrics['delta3']:>8.3f}"
        )

In [ ]:
bm_metrics = []
sgbm_metrics = []
dpt_metrics = []
da_metrics = []

for i, (left_path, right_path, disp_path) in enumerate(triplets):
    # load GT
    disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
    disp_gt[disp_gt == 0] = np.nan
    depth_gt = disp_to_depth(disp_gt, f, B)

    # SGBM
    disp_bm, disp_sgbm = compute_stereo(left_path, right_path)
    depth_bm = disp_to_depth(disp_bm, f, B)
    depth_sgbm = disp_to_depth(disp_sgbm, f, B)
    bm_metrics.append(compute_metrics(depth_bm, depth_gt))
    sgbm_metrics.append(compute_metrics(depth_sgbm, depth_gt))

    # DPT Large
    midas_raw = run_midas(left_path, midas_large, transform, device)
    midas_aligned = median_scale_align(midas_raw, depth_gt)
    dpt_metrics.append(compute_metrics(midas_aligned, depth_gt))

    # DepthAnything
    da_raw = run_depth_anything(left_path, depth_anything)
    da_aligned = median_scale_align(da_raw, depth_gt)
    da_metrics.append(compute_metrics(da_aligned, depth_gt))

    if i % 10 == 0:
        print(f"{i}/200 done")

# print final table
results = {
    'BM': average_metrics(bm_metrics),
    'SGBM': average_metrics(sgbm_metrics),
    'DPT_Large': average_metrics(dpt_metrics),
    'DepthAnything': average_metrics(da_metrics)
}
print_results_table(results)

In [ ]:
def visualize_comparison(triplets, indices, f, B, midas, transform, depth_anything, device):
    fig, axes = plt.subplots(len(indices), 6, figsize=(20, 2*len(indices)))

    for row, i in enumerate(indices):
        left_path, right_path, disp_path = triplets[i]

        # load GT
        disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
        disp_gt[disp_gt == 0] = np.nan
        depth_gt = disp_to_depth(disp_gt, f, B)

        # stereo
        disp_bm, disp_sgbm = compute_stereo(left_path, right_path)
        depth_bm = disp_to_depth(disp_bm, f, B)
        depth_sgbm = disp_to_depth(disp_sgbm, f, B)

        # neural
        midas_raw = run_midas(left_path, midas, transform, device)
        midas_aligned = median_scale_align(midas_raw, depth_gt)

        da_raw = run_depth_anything(left_path, depth_anything)
        da_aligned = median_scale_align(da_raw, depth_gt)

        left_img = cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB)

        axes[row][0].imshow(left_img)
        axes[row][0].set_title(f'Left Image {i}')
        axes[row][1].imshow(depth_gt, cmap='plasma', vmin=0, vmax=80)
        axes[row][1].set_title('GT Depth')
        axes[row][2].imshow(depth_bm, cmap='plasma', vmin=0, vmax=80)
        axes[row][2].set_title('StereoBM')
        axes[row][3].imshow(depth_sgbm, cmap='plasma', vmin=0, vmax=80)
        axes[row][3].set_title('StereoSGBM')
        axes[row][4].imshow(midas_aligned, cmap='plasma', vmin=0, vmax=80)
        axes[row][4].set_title('DPT Large')
        axes[row][5].imshow(da_aligned, cmap='plasma', vmin=0, vmax=80)
        axes[row][5].set_title('DepthAnything')

        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'depth_comparison_5scenes.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
visualize_comparison(triplets, [5, 10, 50, 150, 180], f, B, midas_large, transform, depth_anything, device)

In [ ]:
# Option 1: environment variable
# key = os.environ["WANDB_API_KEY"]
# Option 2: Kaggle secrets
# from kaggle_secrets import UserSecretsClient
# key = UserSecretsClient().get_secret("WANDB_API_KEY")
# Option 3: interactive login
wandb.login()

In [ ]:
# split data
random.seed(42)
all_indices = list(range(200))
random.shuffle(all_indices)
train_indices = all_indices[:160]
val_indices = all_indices[160:]

train_triplets = [triplets[i] for i in train_indices]
val_triplets = [triplets[i] for i in val_indices]

print(f"train: {len(train_triplets)}, val: {len(val_triplets)}")

In [ ]:
class KITTIDepthDataset(Dataset):
    def __init__(self, triplets, transform=None):
        self.triplets = triplets
        self.transform = transform

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        left_path, _, disp_path = self.triplets[idx]

        # load image
        img = cv2.imread(str(left_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # load GT depth
        disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
        disp_gt[disp_gt == 0] = np.nan
        depth_gt = disp_to_depth(disp_gt, f, B)
        depth_gt = np.nan_to_num(depth_gt, nan=0.0)  # replace nan with 0
        depth_gt = cv2.resize(depth_gt, (384, 384), interpolation=cv2.INTER_NEAREST)

        if self.transform:
            img = self.transform(img)

        depth_gt = torch.tensor(depth_gt, dtype=torch.float32)
        return img, depth_gt

In [ ]:
# transforms — resize to MiDaS input size, normalize
# write clean transform because pretrained one adds extra batch dim
train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = KITTIDepthDataset(train_triplets, transform=train_transform)
val_dataset = KITTIDepthDataset(val_triplets, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"train batches: {len(train_loader)}, val batches: {len(val_loader)}")

In [ ]:
# training loop
# load model
model = torch.hub.load("intel-isl/MiDaS", MODEL_NAME)
model.to(device)

criterion = torch.nn.HuberLoss(delta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# init wandb
wandb.init(
    entity='<YOUR_WANDB_ENTITY>',
    project="depth_benchmarking",
    name=f"{MODEL_NAME}-finetuned-kitti",
    config={
        "model": MODEL_NAME,
        "dataset": "KITTI Stereo 2015",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "loss": LOSS,
        "train_size": TRAIN_SIZE,
        "val_size": VAL_SIZE
    }
)

best_val_loss = float('inf')
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()
    train_losses = []

    for imgs, depths in train_loader:
        imgs = imgs.to(device)
        depths = depths.to(device)

        pred = model(imgs)
        if pred.dim() == 3:
            pred = pred.unsqueeze(1)
        pred = torch.nn.functional.interpolate(
            pred, size=depths.shape[1:],
            mode='bicubic', align_corners=False
        ).squeeze(1)

        mask = depths > 0
        if mask.sum() == 0:
            continue

        loss = criterion(pred[mask], depths[mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    val_absrels = []

    with torch.no_grad():
        for imgs, depths in val_loader:
            imgs = imgs.to(device)
            depths = depths.to(device)

            pred = model(imgs)
            if pred.dim() == 3:
                pred = pred.unsqueeze(1)
            pred = torch.nn.functional.interpolate(
                pred, size=depths.shape[1:],
                mode='bicubic', align_corners=False
            ).squeeze(1)

            mask = depths > 0
            if mask.sum() == 0:
                continue

            loss = criterion(pred[mask], depths[mask])
            val_losses.append(loss.item())

            absrel = torch.mean(torch.abs(pred[mask] - depths[mask]) / depths[mask])
            val_absrels.append(absrel.item())

    train_loss = np.mean(train_losses)
    val_loss = np.mean(val_losses)
    val_absrel = np.mean(val_absrels)

    print(f"epoch {epoch+1}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | absrel={val_absrel:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_absrel": val_absrel,
        "learning_rate": optimizer.param_groups[0]['lr']
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  → saved best model at epoch {epoch+1}")

print(f"done. best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")
wandb.log({"best_val_loss": best_val_loss, "best_epoch": best_epoch})
wandb.finish()

In [ ]:
# load fine-tuned model
# uses SAVE_PATH from training above; change if loading weights from elsewhere
finetuned_model = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
finetuned_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
finetuned_model.to(device)
finetuned_model.eval()

# run evaluation
finetuned_metrics = []

for i, (left_path, right_path, disp_path) in enumerate(triplets):
    disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
    disp_gt[disp_gt == 0] = np.nan
    depth_gt = disp_to_depth(disp_gt, f, B)

    finetuned_depth = run_midas_finetuned(left_path, finetuned_model, transform, device)
    # don't need to scale / align bc finetuned model learned to output metric depth
    finetuned_metrics.append(compute_metrics(finetuned_depth, depth_gt))
    if i % 10 == 0:
        print(f"{i}/200 done")

results_finetuned = {
    'DPT_Large_finetuned': average_metrics(finetuned_metrics)
}
print_results_table(results_finetuned)


In [ ]:
# compute depths
left_path, right_path, disp_path = triplets[0]
disp_gt = cv2.imread(str(disp_path), cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0
disp_gt[disp_gt == 0] = np.nan
depth_gt = disp_to_depth(disp_gt, f, B)

finetuned_depth = run_midas_finetuned(left_path, finetuned_model, transform, device)
midas_raw = run_midas(left_path, midas_large, transform, device)
midas_aligned = median_scale_align(midas_raw, depth_gt)
da_raw = run_depth_anything(left_path, depth_anything)
da_aligned = median_scale_align(da_raw, depth_gt)

# visualize
fig, axes = plt.subplots(3, 2, figsize=(20, 12))

axes[0][0].imshow(cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB))
axes[0][0].set_title('Left Image')
axes[0][1].imshow(depth_gt, cmap='plasma', vmin=0, vmax=80)
axes[0][1].set_title('GT Depth')

axes[1][0].imshow(midas_aligned, cmap='plasma', vmin=0, vmax=80)
axes[1][0].set_title('DPT Large (pretrained)')
axes[1][1].imshow(finetuned_depth, cmap='plasma', vmin=0, vmax=80)
axes[1][1].set_title('DPT Large (fine-tuned)')

axes[2][0].imshow(da_aligned, cmap='plasma', vmin=0, vmax=80)
axes[2][0].set_title('DepthAnything V2 Small')
axes[2][1].axis('off')  # empty

for row in axes:
    for ax in row:
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'finetuned_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# workaround for MiDaS patch_size bug during ONNX export
# utils_path points to the cached MiDaS backbones_midas.py
utils_path = os.path.join(torch.hub.get_dir(), 'intel-isl_MiDaS_master', 'midas', 'backbones_midas.py')

with open(utils_path) as f:
    content = f.read()

old = "                    h // pretrained.model.patch_size[1],\n                    w // pretrained.model.patch_size[0],"
new = "                    int(h) // int(pretrained.model.patch_size[1]),\n                    int(w) // int(pretrained.model.patch_size[0]),"

content = content.replace(old, new)

with open(utils_path, 'w') as f:
    f.write(content)

# verify
with open(utils_path) as f:
    content = f.read()
print("patched:", "int(h)" in content)


In [ ]:
torch.hub._load_local  # clear cache

# remove cached midas modules
keys_to_remove = [k for k in sys.modules.keys() if 'midas' in k.lower()]
for k in keys_to_remove:
    del sys.modules[k]

# reload fresh
finetuned_model_cpu = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
finetuned_model_cpu.load_state_dict(
    torch.load(SAVE_PATH, map_location='cpu')
)
finetuned_model_cpu.eval()

# export to ONNX
dummy_input = torch.randn(1, 3, 384, 384)
onnx_path = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned.onnx')

torch.onnx.export(
    finetuned_model_cpu,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    dynamo=False
)

print(f"FP32 size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")


In [ ]:
onnx_path = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned.onnx')
onnx_int8_path = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned_int8.onnx')

quantize_dynamic(
    onnx_path,
    onnx_int8_path,
    weight_type=QuantType.QInt8
)

print(f"FP32 size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")
print(f"INT8 size: {os.path.getsize(onnx_int8_path) / 1e6:.1f} MB")


In [ ]:
onnx_path = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned.onnx')
onnx_int8_path = os.path.join(OUTPUT_PATH, 'dpt_large_finetuned_int8.onnx')

sess_fp32 = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
sess_int8 = ort.InferenceSession(onnx_int8_path, providers=['CPUExecutionProvider'])

dummy_np = np.random.randn(1, 3, 384, 384).astype(np.float32)


In [ ]:
def benchmark(session, input_array, n=10):
    # warmup to avoid cold start effects
    session.run(None, {"input": input_array})
    
    times = []
    for _ in range(n):
        start = time.time()
        session.run(None, {"input": input_array})
        times.append((time.time() - start) * 1000)
    
    return np.mean(times), np.std(times)

In [ ]:
print("benchmarking FP32...")
fp32_mean, fp32_std = benchmark(sess_fp32, dummy_np)
print(f"FP32: {fp32_mean:.1f} ± {fp32_std:.1f} ms")

print("benchmarking INT8...")
int8_mean, int8_std = benchmark(sess_int8, dummy_np)
print(f"INT8: {int8_mean:.1f} ± {int8_std:.1f} ms")

print(f"\nspeedup: {fp32_mean/int8_mean:.2f}x")
print(f"size reduction: {1367.7/351.5:.2f}x")

In [ ]:
all_results = {
    'StereoBM': {
        'mae': 1.201, 'rmse': 3.465, 'absrel': 0.055,
        'delta1': 0.971, 'delta2': 0.985, 'delta3': 0.990
    },
    'StereoSGBM': {
        'mae': 1.175, 'rmse': 3.759, 'absrel': 0.059,
        'delta1': 0.965, 'delta2': 0.983, 'delta3': 0.990
    },
    'DPT_Large_pretrained': {
        'mae': 2.541, 'rmse': 5.475, 'absrel': 0.127,
        'delta1': 0.861, 'delta2': 0.962, 'delta3': 0.986
    },
    'DepthAnything_V2_Small': {
        'mae': 2.154, 'rmse': 5.414, 'absrel': 0.105,
        'delta1': 0.912, 'delta2': 0.976, 'delta3': 0.989
    },
    'DPT_Large_finetuned': {
        'mae': 2.053, 'rmse': 4.366, 'absrel': 0.103,
        'delta1': 0.887, 'delta2': 0.972, 'delta3': 0.992
    }
}

for method, metrics in all_results.items():
    run = wandb.init(
        entity="<YOUR_WANDB_ENTITY>",
        project="depth_benchmarking",
        name=method,
        config={"method": method, 
                "dataset": "KITTI Stereo 2015", 
                "n_images": 200}
    )
    wandb.log(metrics)
    wandb.finish()

In [ ]:
f, B, cx, cy = read_calib_full(calib_path)
print(f"f={f:.2f}, B={B:.4f}, cx={cx:.2f}, cy={cy:.2f}")

In [ ]:
valid = np.isfinite(depth_sgbm) & (depth_sgbm > 0)
Z = depth_sgbm[valid]
uu, vv = np.meshgrid(np.arange(1242), np.arange(375))
X = (uu[valid] - cx) * Z / f
Y = (vv[valid] - cy) * Z / f
print(f"X range: {X.min():.1f} to {X.max():.1f}")
print(f"Y range: {Y.min():.1f} to {Y.max():.1f}")
print(f"Z range: {Z.min():.1f} to {Z.max():.1f}")

In [ ]:
def depth_to_bev(depth, f, cx, cy, grid_res=GRID_RES, x_range=X_RANGE, z_range=Z_RANGE):
    H, W = depth.shape
    
    # pixel coordinates
    u = np.arange(W)
    v = np.arange(H)
    uu, vv = np.meshgrid(u, v)
    
    # valid pixels only
    valid = np.isfinite(depth) & (depth > 0)
    
    Z = depth[valid]
    X = (uu[valid] - cx) * Z / f
    Y = (vv[valid] - cy) * Z / f  # height

    # filter out ground plane and sky
    # in KITTI, camera is ~1.65m above ground
    # Y positive = below camera, Y negative = above camera
    height_mask = (Y > -5) & (Y < 0.5)  # keep points above ground level, removes road / sky
    Z = Z[height_mask]
    X = X[height_mask]
    
    # create grid
    x_bins = int((x_range[1] - x_range[0]) / grid_res)
    z_bins = int((z_range[1] - z_range[0]) / grid_res)
    grid = np.zeros((z_bins, x_bins))
    
    # bin points into grid
    xi = ((X - x_range[0]) / grid_res).astype(int)
    zi = ((Z - z_range[0]) / grid_res).astype(int)
    
    # keep only points within range
    mask = (xi >= 0) & (xi < x_bins) & (zi >= 0) & (zi < z_bins)
    xi, zi = xi[mask], zi[mask]
    
    # mark occupied cells
    grid[zi, xi] = 1
    
    return grid

In [ ]:
# test on scene 0
left_path, right_path, disp_path = triplets[0]
disp_bm, disp_sgbm = compute_stereo(left_path, right_path)
depth_sgbm = disp_to_depth(disp_sgbm, f, B)

bev = depth_to_bev(depth_sgbm, f, cx, cy)

# expand each white region by 2px in all directions
bev_smooth = binary_dilation(bev, iterations=2)

plt.figure(figsize=(8, 12))
plt.imshow(bev_smooth, cmap='gray', origin='lower',
           extent=[X_RANGE[0], X_RANGE[1], Z_RANGE[0], Z_RANGE[1]])
plt.xlabel('X (lateral, meters)')
plt.ylabel('Z (forward, meters)')
plt.title("Bird's Eye View - Scene 0 SGBM")
plt.savefig(os.path.join(OUTPUT_PATH, 'bev.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0, wspace=0.1)

ax1 = fig.add_subplot(gs[0, 0]) # top left: RGB
ax2 = fig.add_subplot(gs[1, 0]) # bottom left: SGBM
ax3 = fig.add_subplot(gs[:, 1]) # right: BEV spanning both rows

ax1.imshow(cv2.cvtColor(cv2.imread(str(left_path)), cv2.COLOR_BGR2RGB))
ax1.set_title('Left Camera Image')
ax1.axis('off')

ax2.imshow(depth_sgbm, cmap='plasma', vmin=0, vmax=80)
ax2.set_title('StereoSGBM Depth Map')
ax2.axis('off')

ax3.imshow(bev_smooth, cmap='Blues', origin='lower', extent=[-25, 35, 3, 80])
ax3.set_xlabel('X - lateral position (meters)', fontsize=12)
ax3.set_ylabel('Z - forward distance (meters)', fontsize=12)
ax3.set_title("Bird's Eye View (top-down occupancy map)", fontsize=12)
ax3.plot(0, 3, 'r^', markersize=15, label='Camera/Car')
ax3.axvline(x=0, color='red', linestyle='--', alpha=0.3, label='Car centerline')
ax3.legend(fontsize=10)
for z in [20, 40, 60]:
    ax3.axhline(y=z, color='blue', linestyle=':', alpha=0.4)
    ax3.text(32, z+1, f'{z}m', color='blue', fontsize=8)

plt.savefig(os.path.join(OUTPUT_PATH, 'l_sgbm_bev.png'), dpi=150, bbox_inches='tight')
plt.show()